In [ ]:
'''
I highly recommend running codes in this file using a high-performance computing system 
since the some original datasets is so large and require high capacity of memory. 
If you run this code in your local environment, it might not work well.
If high perfomance computing systems are not available, please use the cleaned dataset, which is in the same reposytory 
'''

In [ ]:
# install scikit-learn
!pip install scikit-learn

In [1]:
'''
Integrated Industry-Occupation Data:
This script cleans and filters BLS industry-occupation data by removing aggregate totals and formatting numeric values to prepare for analysis.
'''

import pandas as pd
import numpy as np

file_name = 'all_data_M_2024.xlsx'
 
# 1. Load specific columns only to optimize memory usage
target_cols = ['NAICS', 'NAICS_TITLE', 'OCC_CODE', 'OCC_TITLE', 'TOT_EMP', 'PCT_TOTAL','I_GROUP','H_MEAN', 'A_MEAN',]

print("Starting to load data...")
df = pd.read_excel(
    file_name, 
    usecols=target_cols,
    dtype={'NAICS': str, 'OCC_CODE': str},  # Keep codes as strings to preserve leading zeros
    engine='openpyxl'
)

# 2. Clean numeric data
# BLS data often contains '#' or '*' for "too large" or "suppressed" values.
# Convert these to NaN (Not a Number).
df['TOT_EMP'] = pd.to_numeric(df['TOT_EMP'], errors='coerce')
df['PCT_TOTAL'] = pd.to_numeric(df['PCT_TOTAL'], errors='coerce')

# 3. Exclude aggregate rows
# NAICS '0' represents the total for all industries; exclude it for industry-specific analysis.
# OCC_CODE '00-0000' represents the total for all occupations; exclude it.
df_clean = df[
    (df['NAICS'] != '0') & 
    (df['OCC_CODE'] != '00-0000')
].copy()

# 4. Convert percentages to decimals (e.g., 1.5% -> 0.015)
df_clean['WEIGHT'] = df_clean['PCT_TOTAL'] / 100

# Save to CSV at this stage to allow faster loading for future sessions
print(f"\nCleaning complete: {len(df_clean):,} rows of data retained.")
print(df_clean.head())

df_clean.to_csv('bls_staffing_pattern_cleanr1.csv', index=False)

# --- Industry Name List Generation ---
# 1. Get a list of unique industry names (excluding duplicates)
unique_industries = df_clean['NAICS_TITLE'].unique()

# 2. Sort alphabetically for better readability
unique_industries_sorted = sorted([str(i) for i in unique_industries])

# 3. Display the total count and the list of industries
print(f"Number of unique industry names: {len(unique_industries_sorted)}")
print("--- Industry Name List ---")
for industry in unique_industries_sorted:
    print(industry)


データの読み込みを開始します...

クリーニング完了: 413,403 行のデータを保持
    NAICS     NAICS_TITLE         I_GROUP OCC_CODE  \
1  000000  Cross-industry  cross-industry  11-0000   
2  000000  Cross-industry  cross-industry  11-1000   
3  000000  Cross-industry  cross-industry  11-1010   
4  000000  Cross-industry  cross-industry  11-1011   
5  000000  Cross-industry  cross-industry  11-1020   

                         OCC_TITLE     TOT_EMP  PCT_TOTAL  H_MEAN  A_MEAN  \
1           Management Occupations  10966830.0        NaN   68.15  141760   
2                   Top Executives   3822780.0        NaN   67.24  139860   
3                 Chief Executives    211850.0        NaN  126.41  262930   
4                 Chief Executives    211850.0        NaN  126.41  262930   
5  General and Operations Managers   3584420.0        NaN      64  133120   

   WEIGHT  
1     NaN  
2     NaN  
3     NaN  
4     NaN  
5     NaN  
ユニークな産業名の数: 394
--- 産業名一覧 ---
Accommodation
Accommodation and Food Services
Accounting, Tax Pr

In [2]:
'''
Industry-Occupation integrate data: 
This script Processes bls_staffing_pattern_cleanr1.csv to filter out aggregate parent industries 
and retain only the most granular levels of data, saving the result to bls_staffing_pattern_exclude_multipler1.csv.
'''

import pandas as pd

# 1. Load the cleaned dataset and ensure NAICS codes are treated as strings
# Input: 'bls_staffing_pattern_cleanr1.csv'
df_bls_clean = pd.read_csv('bls_staffing_pattern_cleanr1.csv', dtype={'NAICS': str})
df_bls_clean['NAICS'] = df_bls_clean['NAICS'].str.strip()

# Exclude broad categories like "Total all industries" (000000) or "Cross-industry" (000001)
df_bls_clean = df_bls_clean[~df_bls_clean['NAICS'].isin(['000000', '000001'])]

# 2. Identify unique NAICS codes and normalize them by removing trailing zeros
# Example: '211000' -> '211', '211100' -> '2111' to determine hierarchy levels
all_naics = sorted(df_bls_clean['NAICS'].unique())
# Create a dictionary mapping the "stripped" version back to the original code
effective_codes = {code.rstrip('0'): code for code in all_naics}

# 3. Logic to identify and keep only the most detailed ("child") codes
codes_to_keep = []
clean_codes_list = sorted(effective_codes.keys())

for i, code in enumerate(clean_codes_list):
    is_parent = False
    # Check if the current code is a prefix for any subsequent code in the sorted list
    # If it is a prefix, it means a more detailed sub-category exists (it is a parent)
    for next_code in clean_codes_list[i+1:]:
        if next_code.startswith(code):
            is_parent = True
            break
    
    # If no "child" code was found, this is the most granular level; add to keep list
    if not is_parent:
        codes_to_keep.append(effective_codes[code])

# 4. Filter the original dataframe to include only these granular NAICS codes
df_granular_only = df_bls_clean[df_bls_clean['NAICS'].isin(codes_to_keep)].copy()

# 5. Verify the results by comparing row counts
print(f"Original row count: {len(df_bls_clean):,}")
print(f"Granular-only row count: {len(df_granular_only):,}")

# Verify logic using a specific industry example (e.g., Oil and Gas Extraction)
example_check = df_granular_only[df_granular_only['NAICS'].str.startswith('211')]
print("\n--- Processing results for 'Oil and Gas Extraction' sector ---")
print(example_check[['NAICS', 'NAICS_TITLE']].drop_duplicates())

# 6. Save the refined dataset for future analysis
# Output: 'bls_staffing_pattern_exclude_multipler1.csv'
df_granular_only.to_csv('bls_staffing_pattern_exclude_multipler1.csv', index=False)


Original row count: 174,589
Granular-only row count: 102,059

--- Processing results for 'Oil and Gas Extraction' sector ---
        NAICS             NAICS_TITLE
17844  211100  Oil and Gas Extraction


In [7]:
'''
Firm size data:This script extracts national-level employment and payroll data from us_state_6digitnaics_2022.xlsx by filtering out state-specific records 
and saving the result to a smaller CSV file named Employment_and_Payroll_by_industry_by_companysize_susb_united_states_only.csv.
'''
import pandas as pd

# Configuration: Define the input source and the target output file names
input_excel = 'us_state_6digitnaics_2022.xlsx'
output_csv = 'Employment_and_Payroll_by_industry_by_companysize_susb_united_states_only.csv'

print(f"Loading {input_excel}...")

# 1. Read the Excel file, specifying the header row
# We set header=2 to skip the Census Bureau's introductory titles and correctly capture column names.
# All data is read as strings (dtype=str) to prevent numeric conversion of codes.
df = pd.read_excel(input_excel, header=2, dtype=str)

# 2. Extract only the "United States" national-level data
# Filtering out individual state data significantly reduces the dataset size.
print("Filtering for 'United States'...")

# Check if the 'State Name' column exists to perform the filter; otherwise, use the second column as a fallback
if 'State Name' in df.columns:
    df_us = df[df['State Name'].str.strip() == 'United States'].copy()
else:
    # Safety measure: if column names aren't recognized, filter using the index (column 1)
    df_us = df[df.iloc[:, 1].str.strip() == 'United States'].copy()

# 3. Save the filtered national data to a CSV file
# index=False prevents pandas from writing a new row-number column to the file.
# encoding='utf-8-sig' ensures the file is readable by Excel with special characters intact.
print(f"Saving to {output_csv}...")
df_us.to_csv(output_csv, index=False, encoding='utf-8-sig')

print("\nSuccess!")
# Display the column headers that were successfully inherited/extracted
print(f"Inherited column names: {list(df_us.columns)}")


Loading us_state_6digitnaics_2022.xlsx...
Filtering for 'United States'...
Saving to Employment_and_Payroll_by_industry_by_companysize_susb_united_states_only.csv...

Success!
継承された列名: ['State', 'State Name', 'NAICS', 'NAICS Description', 'Enterprise Size', 'Firms', 'Establishments', 'Employment', 'Employment Noise Flag', 'Annual Payroll\n($1,000)', 'Annual Payroll Noise Flag', 'Receipts\n($1,000)', 'Receipts\nNoise Flag']


In [ ]:
'''
Number of firm and employment data(SUSB data):
This script processes a list of annual SUSB Excel files (from 2017 to 2022) to extract national-level data for the "United States" while discarding state-level details, 
resulting in condensed output files with the suffix _US_ONLY.csv
'''

import pandas as pd
import os

# List of source Excel files containing large-scale state and national employment data
large_files = [
    'us_state_6digitnaics_2017.xlsx',
    'us_state_6digitnaics_2018.xlsx',
    'us_state_6digitnaics_2019.xlsx',
    'us_state_6digitnaics_2020.xlsx',
    'us_state_6digitnaics_2021.xlsx',
    'us_state_6digitnaics_2022.xlsx',
]

def extract_us_total(file_list):
     """
    Function to iterate through files, find the header row dynamically,
    and filter for national-level data.
    """
    for f in file_list:
         # Verify if the file exists in the directory to avoid execution errors
        if not os.path.exists(f):
            print(f"File not found: {f}")
            continue
            
        print(f"Processing: {f} ...")
        
        try:
            # Load only the first 20 rows without a header to detect the layout
            # This avoids loading the entire massive file just to find the column names
            df_preview = pd.read_excel(f, nrows=20, header=None)

            # Identify which row contains the actual column titles by searching for "State Name"  
            header_row = 0
            for i, row in df_preview.iterrows():
                if "State Name" in [str(x) for x in row.values]:
                    header_row = i
                    break
            
            print(f"  Detected header at row: {header_row + 1}")
            
            # Load the full dataset starting from the detected header row
            df = pd.read_excel(f, header=header_row, engine='openpyxl')
            
            # Clean column names by replacing newlines with spaces and removing extra whitespace
            df.columns = [str(c).replace('\n', ' ').strip() for c in df.columns]
            
            # Filter the dataframe to keep only the 'United States' totals
            # This filters out the thousands of state-level rows to significantly reduce file size
            if 'State Name' in df.columns:
                df_us_only = df[df['State Name'] == 'United States'].copy()
                
                 # Define the output filename by changing the extension and adding a suffix
                output_fname = f.replace('.xlsx', '_US_ONLY.csv')
                 # Export the filtered data to CSV using 'utf-8-sig' for compatibility with Excel
                df_us_only.to_csv(output_fname, index=False, encoding='utf-8-sig')
                
                print(f"  [Success] Extracted {len(df_us_only)} rows.")
                print(f"  [Saved]   {output_fname}")
            else:
                print(f"  [Error] 'State Name' column not found in {f}")
                
        except Exception as e:
             # Handle unexpected errors (e.g., corrupted files) without stopping the entire loopp
            rint(f"  [Error] Failed to process {f}: {e}")

# Run the extraction process
extract_us_total(large_files)

In [ ]:
'''
Number of firm and employment data:
Standardizes multi-year SUSB datasets by extracting NAICS codes and firm size categories to create a unique identifier (UNIQUE_ID) 
and cleaning key economic metrics for consistent longitudinal analysis.
'''

import pandas as pd
import numpy as np
import os
import re

# =================================================================
# 1. Configuration: Define physical column indices for different data versions
# =================================================================
# SUSB file layouts change over time; these dictionaries map data types to column positions (0-indexed)

# Settings for 2012–2016 files
COLUMN_CONFIG_EARLY_YEARS = {
    "naics_index": 0,        # Col A
    "description_index": 1,  # Col B
    "size_index": 2,         # Col C
    "firms_index": 3,        # Col D
    "employment_index": 5,   # Col F
    "payroll_index": 8,      # Col I
    "receipts_index": 10     # Col K
}

# Settings specifically for 2017 (Column positions shifted for Payroll and Receipts)
COLUMN_CONFIG_YEAR_2017 = {
    "naics_index": 2,        # Col C
    "description_index": 3,  # Col D
    "size_index": 4,         # Col E
    "firms_index": 5,        # Col F
    "employment_index": 7,   # Col H
    "payroll_index": 10,     # Col K
    "receipts_index": 12     # Col M
}

# Settings for 2018–2022 files
COLUMN_CONFIG_LATE_YEARS = {
    "naics_index": 2,        # Col C
    "description_index": 3,  # Col D
    "size_index": 4,         # Col E
    "firms_index": 5,        # Col F
    "employment_index": 7,   # Col H
    "payroll_index": 9,      # Col J
    "receipts_index": 11     # Col L
}

# Mapping firm size numeric codes (e.g., "01" for 0-4 employees) to alphabetical letters for the Unique ID
FIRM_SIZE_DIGIT_TO_LETTER_MAP = {
    "01": "a", "02": "b", "03": "c", "04": "d", "05": "e",
    "06": "f", "07": "g", "08": "h", "09": "i"
}

# Dictionary mapping specific years to their corresponding source data filenames
source_dataset_files_by_year = {
    2012: 'us_6digitnaics_2012.xls', 2013: 'us_6digitnaics_2013.xlsx',
    2014: 'us_6digitnaics_2014.xlsx', 2015: 'us_6digitnaics_2015.xlsx',
    2016: 'us_6digitnaics_2016.xlsx', 2017: 'us_state_6digitnaics_2017_US_ONLY.csv',
    2018: 'us_state_6digitnaics_2018_US_ONLY.csv', 2019: 'us_state_6digitnaics_2019_US_ONLY.csv',
    2020: 'us_state_6digitnaics_2020_US_ONLY.csv', 2021: 'us_state_6digitnaics_2021_US_ONLY.csv',
    2022: 'us_state_6digitnaics_2022_US_ONLY.csv'
}

# =================================================================
# 2. Processing Loop: Iterate through each year and clean the data
# =================================================================
print("--- Generate UNIQUE_ID--------")

for target_year, file_path in source_dataset_files_by_year.items():
    if not os.path.exists(file_path):
        print(f"警告: {file_path} が見つかりません。")
        continue

    # A. Load the file based on its extension (CSV or Excel)
    if file_path.endswith('.csv'):
        current_year_df = pd.read_csv(file_path, dtype=str, header=0)
    else:
        # 2016 has a different header starting position compared to other Excel years
        header_row_index = 7 if target_year == 2016 else 5
        current_year_df = pd.read_excel(file_path, header=header_row_index, dtype=str)

    # Remove any duplicate column headers that might exist in the raw data
    current_year_df = current_year_df.loc[:, ~current_year_df.columns.duplicated()]
    initial_row_count = len(current_year_df)

    # B. Select the appropriate column mapping based on the file's year
    if target_year <= 2016:
        active_column_config = COLUMN_CONFIG_EARLY_YEARS
    elif target_year == 2017:
        active_column_config = COLUMN_CONFIG_YEAR_2017
    else:
        active_column_config = COLUMN_CONFIG_LATE_YEARS

    # C. Data Extraction and Transformation Logic
    # 1. Clean NAICS codes and remove rows ending in '0' (usually aggregate/parent categories)
    raw_naics_series = current_year_df.iloc[:, active_column_config["naics_index"]].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    
    mask_not_ending_in_zero = ~raw_naics_series.str.endswith('0')
    current_year_df_filtered = current_year_df[mask_not_ending_in_zero].copy()
    naics_filtered_series = raw_naics_series[mask_not_ending_in_zero]

    # Create a temporary dataframe to hold the processed columns
    processed_temp_df = pd.DataFrame()
    
    # 2. Standardize NAICS to 6 digits by adding trailing zeros (e.g., '11' becomes '110000')
    processed_temp_df['naics_6digit'] = naics_filtered_series.apply(lambda x: x.ljust(6, '0') if x.isdigit() else np.nan)

    # 3. Extract the numeric firm size code and map it to its corresponding letter
    size_raw_text = current_year_df_filtered.iloc[:, active_column_config["size_index"]].astype(str).str.strip()
    processed_temp_df['size_digit'] = size_raw_text.str.extract(r'^(\d{2})')
    processed_temp_df['FIRM_SIZE_ALPHA'] = processed_temp_df['size_digit'].map(FIRM_SIZE_DIGIT_TO_LETTER_MAP)

    # 4. Generate the UNIQUE_ID by combining the 6-digit NAICS and the firm size letter
    processed_temp_df['UNIQUE_ID'] = processed_temp_df['naics_6digit'] + processed_temp_df['FIRM_SIZE_ALPHA']

    # 5. Extract descriptive attributes
    processed_temp_df['NAICS_CODE'] = naics_filtered_series
    processed_temp_df['DESCRIPTION'] = current_year_df_filtered.iloc[:, active_column_config["description_index"]]
    
    # Define mapping for numeric economic metrics
    metric_column_mapping = {
        "firms_index": "Firms", 
        "employment_index": "Employment", 
        "payroll_index": "Payroll", 
        "receipts_index": "Receipts"
    }

    # Loop through metrics, clean numeric formatting (remove commas), and convert to numeric types
    for config_key, final_name in metric_column_mapping.items():
        column_index = active_column_config[config_key]
        if column_index < current_year_df_filtered.shape[1]:
            raw_values = current_year_df_filtered.iloc[:, column_index].astype(str).str.replace(',', '', regex=False)
            processed_temp_df[final_name] = pd.to_numeric(raw_values, errors='coerce')

    # D. Remove any rows where a Unique ID could not be generated
    processed_temp_df = processed_temp_df.dropna(subset=['UNIQUE_ID']).copy()

    # E. Organize and select final columns for the output file
    final_output_columns = ['UNIQUE_ID', 'NAICS_CODE', 'FIRM_SIZE_ALPHA', 'DESCRIPTION', 'Firms', 'Employment', 'Payroll', 'Receipts']
    current_year_final_df = processed_temp_df[[c for c in final_output_columns if c in processed_temp_df.columns]].copy()

    final_row_count = len(current_year_final_df)

    # F. Save the standardized data to a new CSV file
    output_filename = f"SUSB_Processed_{target_year}.csv"
    current_year_final_df.to_csv(output_filename, index=False, encoding='utf-8-sig')
    
    print(f"  Year {target_year}: Created {output_filename} (Rows: {initial_row_count} -> {final_row_count})")

print("\n--- All file processing tasks completed ---")

In [ ]:
'''
Number of firm and employment data for all year:
merges processed annual SUSB datasets from 2012 to 2022 into a single wide-format panel dataset by linking records 
via a unique identifier and appending yearly suffixes to column names.
'''

import pandas as pd
import os

# =================================================================
# 1. Configuration: Define target years and file parameters
# =================================================================

# List of years in ascending order to ensure columns are organized chronologically from left to right
target_years_ascending = [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

# The common unique identifier used to link rows across different years
primary_merge_key = 'UNIQUE_ID'

# =================================================================
# 2. Execution of Integration Process
# =================================================================

# Initialize an empty variable to hold the combined master dataframe
integrated_wide_dataframe = None

print(f"--- Starting integration using {primary_merge_key} as the key ---")

for current_year in target_years_ascending:
    # Construct the file path for the specific year's processed data
    processed_file_path = f"SUSB_Processed_{current_year}.csv"

    # Check if the file exists; if not, skip this year to avoid errors
    if not os.path.exists(processed_file_path):
        print(f"Warning: {processed_file_path} not found. Skipping.")
        continue

    # Load the processed CSV file, ensuring the ID is read as a string
    current_year_data = pd.read_csv(processed_file_path, dtype={primary_merge_key: str})

    # Rename columns to include the year (e.g., 'Firms' becomes 'Firms_2012'), excluding the UNIQUE_ID
    current_year_data = current_year_data.rename(
        columns={col: f"{col}_{current_year}" for col in current_year_data.columns if col != primary_merge_key}
    )

    if integrated_wide_dataframe is None:
        # Initialize the master dataframe with the first available year (2012)
        integrated_wide_dataframe = current_year_data
        print(f"Loaded base data for the year {current_year}.")
    else:
         # Perform a 'right' join to merge previous years into the current (newer) year's data
        # This preserves IDs found in the latest dataset while attaching historical data
        integrated_wide_dataframe = pd.merge(
            integrated_wide_dataframe,
            current_year_data,
            on=primary_merge_key,
            how='right'
        )
        print(f" -> Right-joined data for the year {current_year}.")

# =================================================================
# 3. Final Data Organization and Export
# =================================================================

# Sort the final dataset by UNIQUE_ID and reset the index for a clean output
integrated_wide_dataframe = integrated_wide_dataframe.sort_values(primary_merge_key).reset_index(drop=True)

print(f"\nIntegration complete.")
print(f"Final row count: {len(integrated_wide_dataframe)}")
print(f"Final column count: {len(integrated_wide_dataframe.columns)}")

# Save the resulting wide-format panel data to a new CSV file
output_final_csv = "SUSB_Integrated_Panel_Wide_2012_2022_v7.csv"
integrated_wide_dataframe.to_csv(output_final_csv, index=False, encoding='utf-8-sig')

print(f"Result saved to {output_final_csv}.")

# Display the first few rows of the integrated data for verification
display(integrated_wide_dataframe.head())

In [ ]:
'''
Number of firm and employment data for all year:
Filters the integrated SUSB panel data to isolate records for "Total" (a), "SME" (h), and "Large" (i) firm size categories 
by analyzing the suffix of each Unique ID.
'''

import pandas as pd

# =================================================================
# 1. Configuration: Define input and output files
# =================================================================
# Load the integrated wide-format panel data created in the previous step
input_integrated_csv = "SUSB_Integrated_Panel_Wide_2012_2022_v7.csv"
output_filtered_csv = "SUSB_Final_Regression_Data_a_h_i_Only.csv"

print(f"Loading {input_integrated_csv}...")

# Load the dataset while ensuring UNIQUE_ID remains a string to preserve leading zeros
integrated_dataframe = pd.read_csv(input_integrated_csv, dtype={'UNIQUE_ID': str})

# =================================================================
# 2. Filtering Process (Extracting only 'a', 'h', and 'i' suffixes)
# =================================================================
print("Filtering data for Size 'a'(total),'h' (<500) and 'i' (500+)...")

# Define the suffixes representing the target firm size categories
# 'a' = Total, 'h' = SMEs (<500 employees), 'i' = Large firms (500+ employees)
target_size_indicators = ('a','h', 'i')

# Filter rows where the UNIQUE_ID ends with one of the target indicators
# str.endswith identifies the alphabet at the end of the ID generated in previous scripts
filtered_regression_dataframe = integrated_dataframe[
    integrated_dataframe['UNIQUE_ID'].str.endswith(target_size_indicators)
].copy()

# =================================================================
# 3. Data Organization and Export
# =================================================================

# Sort the data by UNIQUE_ID to group NAICS codes and sizes logically
filtered_regression_dataframe = filtered_regression_dataframe.sort_values('UNIQUE_ID').reset_index(drop=True)

print(f"\nFiltering complete.")
print(f"Rows before filtering: {len(integrated_dataframe)}")
print(f"Rows after filtering: {len(filtered_regression_dataframe)}")
print(f"Retained size categories: {target_size_indicators}")

# Save the subset to a new CSV file for regression analysis
filtered_regression_dataframe.to_csv(output_filtered_csv, index=False, encoding='utf-8-sig')

print(f"Result saved to {output_filtered_csv}.")

# Display the first 10 rows for verification
display(filtered_regression_dataframe.head(10))